# 12. 전이학습으로 꽃 분류 개선하기

이번 장에서는 CNN을 처음부터 학습하는 대신, 이미 많은 이미지를 보고 배운 모델을 가져와서 꽃 분류에 사용합니다.

이 방법을 전이학습이라고 합니다.

핵심 생각은 다음과 같습니다.

```text
이미지를 보는 기본 능력은 빌려오고,
꽃 이름을 고르는 마지막 판단 부분만 새로 배운다.
```

## 1. 라이브러리 불러오기

이번 장에서는 Keras에 포함된 `MobileNetV2`를 사용합니다.

MobileNetV2는 비교적 가벼운 이미지 분류 모델이며, 전이학습 입문용으로 자주 사용됩니다.

In [ ]:
# pathlib.Path는 파일과 폴더 경로를 다루기 위한 도구입니다.
from pathlib import Path

# matplotlib.pyplot은 학습 곡선과 이미지를 그릴 때 사용합니다.
import matplotlib.pyplot as plt

# tensorflow는 데이터셋 생성과 softmax 계산 등에 사용합니다.
import tensorflow as tf

# Sequential은 층을 순서대로 쌓는 Keras 모델 방식입니다.
from keras.models import Sequential

# Input은 모델이 받을 입력 모양을 알려주는 층입니다.
# Dense는 최종 판단을 담당하는 완전연결층입니다.
# Dropout은 과적합을 줄이기 위해 일부 출력을 학습 중 임시로 끄는 층입니다.
# GlobalAveragePooling2D는 특징 맵을 평균으로 요약해 1차원 벡터로 만듭니다.
from keras.layers import Input, Dense, Dropout, GlobalAveragePooling2D

# RandomFlip, RandomRotation, RandomZoom은 이미지 데이터 증강에 사용합니다.
from keras.layers import RandomFlip, RandomRotation, RandomZoom

# MobileNetV2는 사전학습된 이미지 모델입니다.
# preprocess_input은 MobileNetV2가 기대하는 방식으로 픽셀값을 바꾸는 함수입니다.
from keras.applications import MobileNetV2
from keras.applications.mobilenet_v2 import preprocess_input

## 2. 데이터 경로와 기본 설정

10장, 11장과 같은 꽃 이미지 데이터를 사용합니다.

전이학습 모델은 보통 특정 입력 크기를 기대합니다. 여기서는 `160 x 160`으로 맞춥니다.

In [ ]:
# 꽃 이미지 데이터가 들어 있는 폴더입니다.
DATA_DIR = Path(r"C:\work\deepLearning\flower_dataset\flower_photos")

# 모델을 저장할 폴더입니다.
MODEL_DIR = Path(r"C:\work\deepLearning\model")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 이미지 크기입니다.
# MobileNetV2는 다양한 크기를 받을 수 있지만, CPU 환경을 고려해 160x160으로 둡니다.
IMG_SIZE = (160, 160)

# 배치 크기입니다. 한 번에 32장씩 모델에 넣습니다.
BATCH_SIZE = 32

# 학습/검증 분리가 매번 같도록 seed를 고정합니다.
SEED = 42

print("데이터 폴더 존재 여부:", DATA_DIR.exists())

## 3. 학습 데이터와 검증 데이터 만들기

`image_dataset_from_directory`는 폴더 이름을 정답 이름으로 사용해 이미지 데이터셋을 만듭니다.

이번에도 전체 데이터 중 20%를 검증용으로 나눕니다.

In [ ]:
# 학습용 데이터셋입니다.
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# 검증용 데이터셋입니다.
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# 폴더 이름에서 읽어 온 클래스 이름입니다.
class_names = train_ds.class_names
num_classes = len(class_names)

print("클래스 이름:", class_names)
print("클래스 개수:", num_classes)

## 4. 데이터셋 성능 설정

`cache()`와 `prefetch()`는 데이터를 더 효율적으로 공급하기 위한 설정입니다.

처음에는 “이미지를 미리 준비해서 학습 흐름을 덜 끊기게 한다” 정도로 이해하면 됩니다.

In [ ]:
# TensorFlow가 현재 환경에 맞게 적절한 병렬 처리 값을 고르게 합니다.
AUTOTUNE = tf.data.AUTOTUNE

# cache()는 한 번 읽은 데이터를 다시 사용할 때 속도를 높이는 데 도움을 줍니다.
# prefetch()는 모델이 현재 배치를 학습하는 동안 다음 배치를 미리 준비합니다.
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

## 5. 데이터 증강 만들기

전이학습에서도 데이터 증강은 도움이 됩니다.

이미지를 조금 뒤집거나 회전하거나 확대해서 모델이 더 다양한 상황을 보게 합니다.

In [ ]:
data_augmentation = Sequential([
    # 좌우 반전을 랜덤하게 적용합니다.
    RandomFlip("horizontal"),
    
    # 이미지를 조금 회전합니다.
    RandomRotation(0.1),
    
    # 이미지를 조금 확대하거나 축소합니다.
    RandomZoom(0.1),
], name="data_augmentation")

## 6. MobileNetV2 불러오기

여기서 중요한 옵션은 세 가지입니다.

```text
weights="imagenet"   : ImageNet이라는 큰 이미지 데이터로 미리 학습된 가중치를 사용한다.
include_top=False    : 원래의 1000종 분류기는 떼어낸다.
input_shape=...      : 우리가 넣을 이미지 모양을 알려준다.
```

처음 실행할 때는 사전학습 가중치를 다운로드할 수 있습니다.

In [ ]:
# MobileNetV2의 특징 추출 부분을 불러옵니다.
base_model = MobileNetV2(
    weights="imagenet",        # ImageNet으로 미리 학습된 가중치를 사용합니다.
    include_top=False,         # 원래의 마지막 분류층은 제거합니다.
    input_shape=IMG_SIZE + (3,) # 입력 이미지 모양입니다. 컬러 이미지라 채널 수 3을 붙입니다.
)

# trainable=False는 base_model의 기존 가중치를 학습 중 바꾸지 않겠다는 뜻입니다.
# 즉, 이미 배운 이미지 특징 추출 능력은 그대로 둡니다.
base_model.trainable = False

print("MobileNetV2 층 개수:", len(base_model.layers))
print("base_model 학습 가능 여부:", base_model.trainable)

## 7. 전이학습 모델 만들기

모델 구조는 다음과 같습니다.

```text
입력 이미지
-> 데이터 증강
-> MobileNetV2 전용 전처리
-> MobileNetV2 특징 추출
-> GlobalAveragePooling2D
-> Dropout
-> Dense 출력층
```

마지막 출력층의 뉴런 수는 꽃 클래스 개수와 같아야 합니다.

In [ ]:
model = Sequential([
    # 입력 이미지 모양을 명시합니다.
    Input(shape=IMG_SIZE + (3,)),
    
    # 학습 중 이미지에 랜덤 변형을 적용합니다.
    data_augmentation,
    
    # MobileNetV2가 기대하는 픽셀값 범위로 변환합니다.
    # 10장의 Rescaling(1/255)와 다르게, 사전학습 모델마다 권장 전처리가 따로 있을 수 있습니다.
    preprocess_input,
    
    # 이미 학습된 MobileNetV2가 이미지 특징을 추출합니다.
    base_model,
    
    # 특징 맵 전체를 평균으로 요약합니다.
    # Flatten보다 파라미터 수를 줄이는 데 도움이 됩니다.
    GlobalAveragePooling2D(),
    
    # 과적합을 줄이기 위해 일부 출력을 학습 중 임시로 끕니다.
    Dropout(0.3),
    
    # 꽃 종류별 점수를 출력합니다.
    Dense(num_classes)
])

model.summary()

## 8. 모델 컴파일

정답은 정수 번호 형태입니다.

따라서 다중 분류용 손실 함수인 `SparseCategoricalCrossentropy`를 사용합니다.

마지막 Dense 층이 softmax를 직접 쓰지 않으므로 `from_logits=True`를 사용합니다.

In [ ]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

model.compile(
    # Adam은 많이 사용되는 최적화 방법입니다.
    optimizer="adam",
    
    # 정수 라벨 다중 분류 손실 함수입니다.
    loss=loss_fn,
    
    # 전체 중 맞힌 비율을 확인합니다.
    metrics=["accuracy"]
)

## 9. 모델 학습

전이학습은 처음부터 CNN을 학습하는 것보다 빠르게 좋은 출발점을 가질 수 있습니다.

다만 현재 환경을 고려해 epoch는 작게 시작합니다.

In [ ]:
EPOCHS = 3

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

## 10. 학습 곡선 확인

처음부터 만든 CNN과 마찬가지로 학습 정확도와 검증 정확도를 함께 봐야 합니다.

검증 정확도가 더 안정적으로 올라가는지 확인합니다.

In [ ]:
history_dict = history.history

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history_dict["accuracy"], label="train accuracy")
plt.plot(history_dict["val_accuracy"], label="validation accuracy")
plt.title("Transfer Learning Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_dict["loss"], label="train loss")
plt.plot(history_dict["val_loss"], label="validation loss")
plt.title("Transfer Learning Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

## 11. 예측 결과 보기

모델의 출력은 클래스별 점수입니다.

`softmax`를 적용하면 가장 가능성이 높은 꽃 이름과 확신도를 볼 수 있습니다.

In [ ]:
plt.figure(figsize=(10, 10))

for images, labels in val_ds.take(1):
    logits = model.predict(images, verbose=0)
    
    # 클래스별 점수를 확률처럼 해석 가능한 값으로 바꿉니다.
    probabilities = tf.nn.softmax(logits, axis=1)
    
    for i in range(9):
        pred_index = int(tf.argmax(probabilities[i]).numpy())
        true_index = int(labels[i].numpy())
        confidence = float(probabilities[i][pred_index].numpy())
        
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(
            f"예측: {class_names[pred_index]}\n"
            f"정답: {class_names[true_index]}\n"
            f"확신: {confidence:.2f}"
        )
        plt.axis("off")

plt.tight_layout()
plt.show()

## 12. 모델 저장

전이학습 모델도 저장해 둡니다.

나중에 11장처럼 오답 분석을 하거나, 데모 앱에서 불러와 사용할 수 있습니다.

In [ ]:
model_path = MODEL_DIR / "flower_transfer_learning_mobilenetv2.keras"

# save()는 모델 구조와 학습된 가중치를 함께 저장합니다.
model.save(model_path)

print("저장 완료:", model_path)

## 13. 처음부터 만든 CNN과 비교하기

비교할 때는 단순히 정확도만 보지 않습니다.

아래 질문을 함께 생각합니다.

```text
어느 모델이 더 빨리 학습되는가?
어느 모델이 검증 정확도가 더 안정적인가?
어느 모델이 과적합이 덜한가?
어느 모델이 설명하기 쉬운가?
포트폴리오에서는 어떤 흐름으로 보여주는 것이 좋은가?
```

In [ ]:
print("비교 메모")
print("- 처음부터 만든 CNN: 구조를 이해하기 좋다.")
print("- 전이학습 모델: 작은 데이터셋에서 좋은 출발점을 가질 수 있다.")
print("- 포트폴리오 흐름: 기본 CNN -> 오답 분석 -> 전이학습 개선 순서가 자연스럽다.")

## 14. 이번 장 정리

이번 장에서 배운 핵심은 다음과 같습니다.

```text
1. 전이학습은 이미 배운 이미지 특징 추출 능력을 가져와 쓰는 방법이다.
2. include_top=False는 원래 모델의 마지막 분류기를 떼어내겠다는 뜻이다.
3. base_model.trainable=False는 기존 특징 추출 부분을 얼려 두겠다는 뜻이다.
4. MobileNetV2는 꽃 이미지의 특징을 추출하고, 새 Dense 층이 꽃 이름을 판단한다.
5. 작은 데이터셋에서는 처음부터 큰 모델을 학습하는 것보다 전이학습이 유리할 수 있다.
```

## 과제

1. 전이학습을 자기 말로 설명해보세요.
2. `include_top=False`가 왜 필요한지 적어보세요.
3. `base_model.trainable=False`가 무슨 뜻인지 적어보세요.
4. 처음부터 만든 CNN과 전이학습 모델을 포트폴리오에서 어떤 순서로 설명할지 문장으로 써보세요.